# Lab 6 Report: 
## Text Classification on AG News Dataset

In [3]:
import torch
import time
import numpy as np
import seaborn as sns
import torchtext

In [ ]:
from IPython.display import Image # For displaying images in colab jupyter cell


In [ ]:
Image('lab6_exercise.png', width = 1000)


In [4]:
## <span style='color:red'> AG News Text Classification </span>

In [ ]:
#!pip install torchtext==0.17.2
#!pip install torchdata==0.7.1


In [5]:
# Seaborn plot styling
sns.set(style = 'white', font_scale = 2)

## Prepare Data

In [ ]:
from pathlib import Path
import csv
from collections import Counter
from torch.utils.data import Dataset, DataLoader


POSSIBLE_DATA_ROOTS = [
    Path('datasets/AG_NEWS'),
    Path('Lab_6/datasets/AG_NEWS'),
    Path('Lab_6/datasets/datasets/AG_NEWS'),
]


def _find_dataset_root():
    for candidate in POSSIBLE_DATA_ROOTS:
        if (candidate / 'train.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find the AG_NEWS CSV files. Run the Lab 6 sync step to populate datasets/AG_NEWS.')


def _load_split(root: Path, split: str):
    csv_path = root / f"{split}.csv"
    samples = []
    with csv_path.open(newline='', encoding='utf-8') as handle:
        reader = csv.reader(handle)
        for label, title, description in reader:
            text = f"{title} {description}".replace('\n', ' ').strip()
            samples.append((int(label), text))
    return samples


data_root = _find_dataset_root()
train_samples = _load_split(data_root, 'train')
test_samples = _load_split(data_root, 'test')


class AGNewsDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

    def __iter__(self):
        return iter(self.samples)


train_dataset = AGNewsDataset(train_samples)
test_dataset = AGNewsDataset(test_samples)
train_iter = list(train_dataset)
test_iter = list(test_dataset)

train_label_counts = Counter(label for label, _ in train_iter)
test_label_counts = Counter(label for label, _ in test_iter)
print(f'Train label distribution: {train_label_counts}')
print(f'Test label distribution: {test_label_counts}')
print(f'Loaded {len(train_dataset)} training samples and {len(test_dataset)} test samples from {data_root}')

Train label distribution: Counter({3: 30000, 4: 30000, 2: 30000, 1: 30000})
Test label distribution: Counter({3: 1900, 4: 1900, 2: 1900, 1: 1900})
Loaded 120000 training samples and 7600 test samples from datasets/AG_NEWS


In [ ]:
# Reload train_iter for use in DataLoader
tokenizer = torchtext.data.utils.get_tokenizer("basic_english")

def yield_tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text)

# Create vocabulary with special tokens for padding and unknown words
# For torchtext 0.17.2, use build_vocab_from_iterator which supports specials parameter
# Define special tokens
specials = ["<unk>", "<pad>"]

# Create vocab with special tokens using build_vocab_from_iterator
vocab = torchtext.vocab.build_vocab_from_iterator(yield_tokens(train_iter), specials=specials)
vocab.set_default_index(vocab["<unk>"])


TypeError: Vocab.__init__() got an unexpected keyword argument 'specials'

In [ ]:
def text_pipeline(x): 
    return vocab(tokenizer(x))
    
# Example: Test the pipeline on a sample text
sample_text = "This is a sample news article!"
print(text_pipeline(sample_text))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Function to calculate the length of each text in the dataset
def get_text_lengths(data_iter):
    lengths = []
    for _, text in tqdm(data_iter):
        tokenized_text = tokenizer(text)
        lengths.append(len(tokenized_text))
    return np.array(lengths)

# Get text lengths from the entire dataset
train_text_lengths = get_text_lengths(train_iter)
test_text_lengths = get_text_lengths(test_iter)
# Plot the distribution of text lengths

# Define bin edges using the range of both datasets
min_bin = min(train_text_lengths.min(), test_text_lengths.min())
max_bin = max(train_text_lengths.max(), test_text_lengths.max())
bins = np.linspace(min_bin, max_bin, 30)  # Adjust number of bins as needed

plt.figure(figsize=(10, 6))
plt.subplot(2,1,1)
sns.histplot(train_text_lengths, kde=True, bins=50)
plt.xlabel('Train Text Length')
plt.ylabel('Frequency')
plt.subplot(2,1,2)
sns.histplot(test_text_lengths, kde=True, bins=50)
plt.xlabel('Test Text Length')
plt.ylabel('Frequency')
plt.suptitle('Distribution of Text Lengths in the AG News Dataset')
plt.show()


In [ ]:
from torch.nn.utils.rnn import pad_sequence

# Define collate function for padding and batching
# setting a max_seq_len helps with estimating the max gpu memory usage
def collate_batch(batch, max_seq_len=1024):
    labels, texts = zip(*batch)
    # the labels start at 1 but predictions start at 0. To align them, we modify lables
    labels = torch.tensor(labels,dtype=torch.long)-1
    
    text_list = []
    for text in texts:
        # Truncate or pad to max_seq_len
        tokenized_text = text_pipeline(text)
        if len(tokenized_text) > max_seq_len:
            tokenized_text = tokenized_text[:max_seq_len]  # Truncate if longer than max_seq_len
        else:
            # Pad if shorter than max_seq_len
            tokenized_text = tokenized_text + [vocab["<pad>"]] * (max_seq_len - len(tokenized_text))
        
        text_list.append(torch.tensor(tokenized_text, dtype=torch.long))
    
    padded_texts = torch.stack(text_list)  # Stack the sequences into a tensor
    return padded_texts, labels


In [ ]:
from torch.utils.data import Dataset

train_dataset = AGNewsDataset(train_samples)
test_dataset = AGNewsDataset(test_samples)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

# Train_loader and test_loader sanity check
# Initialize counter
label_counter = Counter()

# Iterate through batches in train_loader
for texts, labels in train_loader:
    label_counter.update(labels.tolist())
print("Label counts:", label_counter)


## Define Model

In [ ]:
class TransformerModel(torch.nn.Module):
    def __init__(self, vocab_size, embed_size, num_heads, num_encoder_layers, num_classes, dropout=0.1):
        
        super(TransformerModel, self).__init__()
        
        self.embedding = torch.nn.Embedding(vocab_size, embed_size)
        self.transformer = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(embed_size, num_heads, embed_size * 2, dropout),
            num_encoder_layers
        )
        self.fc = torch.nn.Linear(embed_size, num_classes)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x):
        x = self.embedding(x)  # Embedding layer
        x = x.permute(1, 0, 2)  # Transformer expects (seq_len, batch_size, embedding_size)
        x = self.transformer(x)  # Apply transformer
        x = x.mean(dim=0)  # Pooling (take the mean of all tokens in the sequence)
        x = self.dropout(x)
        x = self.fc(x)  # Final classification layer
        return x

## Define Hyperparameters

In [ ]:
# Check for device compatibility, prioritizing CUDA, then MPS for MacBooks with Apple Silicon, and defaulting to CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Initialize the model, 
embed_size = 32
num_heads = 4
num_encoder_layers = 2
num_classes = 4  # World, Sports, Business, Sci/Tech
model = TransformerModel(len(vocab), embed_size, num_heads, num_encoder_layers, num_classes)

# Initialize loss function, and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.to(device)

## Identify Tracked values

In [ ]:
num_epochs = 5
train_losses = np.zeros(num_epochs)
train_accuracies = np.zeros(num_epochs)

test_losses = np.zeros(num_epochs)
test_accuracies = np.zeros(num_epochs)

In [ ]:
def train_epoch(model, train_loader, loss_fn, optimizer):
    model.train()
    epoch_loss = 0
    epoch_accuracy = 0
    # total_batches = 0
    total_batches = len(train_loader)
    
    for texts, labels in tqdm(train_loader):
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(texts)
        
        # Compute loss and gradients
        loss = loss_fn(outputs, labels)
        loss.backward()
        
        # Update model parameters
        optimizer.step()
        
        # Calculate accuracy
        preds = torch.argmax(outputs, dim=1)
        correct = (preds == labels).sum().item()
        accuracy = correct / labels.size(0)
        
        epoch_loss += loss.item()
        epoch_accuracy += accuracy
    
    return epoch_loss / total_batches, epoch_accuracy / total_batches

def evaluate(model, test_loader, loss_fn):
    model.eval()
    epoch_loss = 0
    epoch_accuracy = 0
    total_batches = len(test_loader)
    
    with torch.no_grad():
        for texts, labels in tqdm(test_loader):
            texts, labels = texts.to(device), labels.to(device)
            # Forward pass
            outputs = model(texts)
            
            # Compute loss
            loss = loss_fn(outputs, labels)
            
            # Calculate accuracy
            preds = torch.argmax(outputs, dim=1)
            correct = (preds == labels).sum().item()
            accuracy = correct / labels.size(0)
            
            epoch_loss += loss.item()
            epoch_accuracy += accuracy
    
    return epoch_loss / total_batches, epoch_accuracy / total_batches


In [ ]:
for epoch in range(num_epochs):
    start_time = time.time()
    
    # Train for one epoch
    train_loss, train_accuracy = train_epoch(model, train_loader, loss_fn, optimizer)
    train_losses[epoch] = train_loss
    train_accuracies[epoch] = train_accuracy
    
    # Evaluate on the test set
    test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)
    test_losses[epoch] = test_loss
    test_accuracies[epoch] = test_accuracy
    end_time = time.time()
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | Time: {end_time - start_time:.2f}s")
    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f}")


## Train Model

In [ ]:
# Collect predictions and identify correct/incorrect classifications
import numpy as np  # Required for .numpy() tensor method
model.eval()
all_predictions = []
all_labels = []
all_texts = []

# Map class indices to names (model uses 0-3, original labels are 1-4)
class_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

with torch.no_grad():
    for texts, labels in test_loader:
        texts, labels = texts.to(device), labels.to(device)
        outputs = model(texts)
        preds = torch.argmax(outputs, dim=1)
        
        # Store predictions, labels, and texts
        all_predictions.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
        # Store original text samples (need to get them from the batch)
        for i in range(len(labels)):
            # Find the corresponding text in test_samples
            # Since test_loader uses shuffle=False, we can track index
            pass

# Get texts from test_samples directly
# We need to match predictions with original texts
# Since DataLoader processes in batches, we'll iterate through test_dataset
all_texts = []
all_original_labels = []
all_preds = []
all_model_labels = []

# Iterate through test dataset and get predictions
model.eval()
idx = 0
with torch.no_grad():
    for batch_texts, batch_labels in test_loader:
        batch_texts = batch_texts.to(device)
        outputs = model(batch_texts)
        batch_preds = torch.argmax(outputs, dim=1).cpu().numpy()
        
        # Get original samples for this batch
        for i in range(len(batch_labels)):
            original_label, original_text = test_samples[idx]
            all_texts.append(original_text)
            all_original_labels.append(original_label)  # Original label (1-4)
            all_preds.append(batch_preds[i])  # Model prediction (0-3)
            all_model_labels.append(batch_labels[i].item())  # Model label (0-3)
            idx += 1

# Convert original labels to model labels (1-4 -> 0-3) for comparison
correct_predictions = []
incorrect_predictions = []

for i in range(len(all_texts)):
    original_label = all_original_labels[i]  # 1-4
    model_label = original_label - 1  # Convert to 0-3
    pred = all_preds[i]  # 0-3
    
    if pred == model_label:
        correct_predictions.append({
            'text': all_texts[i],
            'true_label': original_label,
            'predicted_label': pred + 1,  # Convert back to 1-4 for display
            'true_class': class_names[model_label],
            'predicted_class': class_names[pred]
        })
    else:
        incorrect_predictions.append({
            'text': all_texts[i],
            'true_label': original_label,
            'predicted_label': pred + 1,  # Convert back to 1-4 for display
            'true_class': class_names[model_label],
            'predicted_class': class_names[pred]
        })

# Print 3 correctly classified examples
print("=" * 80)
print("3 EXAMPLES OF CORRECTLY CLASSIFIED ARTICLES")
print("=" * 80)
for i, example in enumerate(correct_predictions[:3], 1):
    print(f"\nExample {i}:")
    print(f"True Label: {example['true_label']} ({example['true_class']})")
    print(f"Predicted Label: {example['predicted_label']} ({example['predicted_class']})")
    # Print snippet (first 200 characters)
    snippet = example['text'][:200] + "..." if len(example['text']) > 200 else example['text']
    print(f"Text Snippet: {snippet}")
    print("-" * 80)

# Print 3 incorrectly classified examples
print("\n" + "=" * 80)
print("3 EXAMPLES OF INCORRECTLY CLASSIFIED ARTICLES")
print("=" * 80)
for i, example in enumerate(incorrect_predictions[:3], 1):
    print(f"\nExample {i}:")
    print(f"True Label: {example['true_label']} ({example['true_class']})")
    print(f"Predicted Label: {example['predicted_label']} ({example['predicted_class']})")
    # Print snippet (first 200 characters)
    snippet = example['text'][:200] + "..." if len(example['text']) > 200 else example['text']
    print(f"Text Snippet: {snippet}")
    print("-" * 80)


In [ ]:
def compute_accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    predictions = logits.argmax(dim=1)
    correct = (predictions == labels).sum().item()
    return correct / labels.size(0)


def train_one_epoch(epoch_index: int) -> tuple[float, float]:
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_examples = 0

    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch_index + 1}/{num_epochs} [train]', leave=False)
    for inputs, labels in progress_bar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        loss = loss_fn(logits, labels)
        loss.backward()
        if gradient_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)
        optimizer.step()

        batch_size = labels.size(0)
        running_examples += batch_size
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == labels).sum().item()

        progress_bar.set_postfix({
            'loss': running_loss / running_examples,
            'acc': running_correct / running_examples,
        })

    epoch_loss = running_loss / running_examples
    epoch_accuracy = running_correct / running_examples
    return epoch_loss, epoch_accuracy


def evaluate_model() -> tuple[float, float]:
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_examples = 0

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc='Evaluating', leave=False):
            inputs = inputs.to(device)
            labels = labels.to(device)
            logits = model(inputs)
            loss = loss_fn(logits, labels)

            batch_size = labels.size(0)
            running_examples += batch_size
            running_loss += loss.item() * batch_size
            running_correct += (logits.argmax(dim=1) == labels).sum().item()

    epoch_loss = running_loss / running_examples
    epoch_accuracy = running_correct / running_examples
    return epoch_loss, epoch_accuracy


best_state = None
best_test_accuracy = 0.0

for epoch in range(num_epochs):
    start_time = time.time()

    train_loss, train_accuracy = train_one_epoch(epoch)
    test_loss, test_accuracy = evaluate_model()
    scheduler.step()

    train_losses[epoch] = train_loss
    train_accuracies[epoch] = train_accuracy
    test_losses[epoch] = test_loss
    test_accuracies[epoch] = test_accuracy
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'learning_rate': scheduler.get_last_lr()[0],
        'epoch_time_sec': time.time() - start_time,
    })

    if test_accuracy > best_test_accuracy:
        best_test_accuracy = test_accuracy
        best_state = {
            'model_state': copy.deepcopy(model.state_dict()),
            'epoch': epoch,
        }

    print(f"Epoch {epoch + 1:02d}/{num_epochs} | {history[-1]['epoch_time_sec']:.1f}s")
    print(f"  Train    - loss: {train_loss:.4f}, acc: {train_accuracy:.4f}")
    print(f"  Validate - loss: {test_loss:.4f}, acc: {test_accuracy:.4f}")
    print(f"  LR: {history[-1]['learning_rate']:.6f}")

if best_state is not None and best_state['epoch'] != num_epochs - 1:
    print(f"Restoring best model weights from epoch {best_state['epoch'] + 1}")
    model.load_state_dict(best_state['model_state'])

## Visualize and Evaluate Model

In [ ]:
num_epochs = 5
epochs = list(range(0, num_epochs))

# Create 2x2 grid of subplots using plt.subplot
plt.figure(figsize=(10, 8))

# Training Loss
plt.subplot(2, 2, 1)  # (rows, columns, index)
plt.plot(epochs, train_losses, color='blue', label='Train Loss')
plt.title('Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Testing Loss
plt.subplot(2, 2, 2)
plt.plot(epochs, test_losses, color='orange', label='Test Loss')
plt.title('Testing Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Training Accuracy
plt.subplot(2, 2, 3)
plt.plot(epochs, train_accuracies, color='green', label='Train Accuracy')
plt.title('Training Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Testing Accuracy
plt.subplot(2, 2, 4)
plt.plot(epochs, test_accuracies, color='red', label='Test Accuracy')
plt.title('Testing Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Adjust layout and display
plt.tight_layout()
plt.show()

# Notes
- Not keeping a clean venv really bit me with torchtext issues
- The example this time wasn't as helpful this time around
- This felt like an order of magnitude more complicated than previous labs with prepocessing and multiple categories
- I had to lean on AI tools (Cursor) heavily for the printing and error correction
- It seems to be overtrained on Sci/Tech, predicting it more often
- Itterating on changes are unfeasable due to the long training times (~15 mins with 2080ti)
- I think I could have increased the embeds to 64 or even 128, my graphics card could have handled it and it probably would have made better results
